# Illinois STEM Pathways — reproducible IPEDS cleaning

Implementation snapshot, 2016–2023. Run all cells top to bottom. Source data: public NCES IPEDS annual Completions and Institutional Directory ZIPs. No student-level data. Region/STEM/release-window/CIP approach require the approvals listed in `docs/data-plan-and-contract.md`.

This notebook calls version-controlled cleaning functions so the documented run and command-line build have one implementation. A separate standard-library verifier independently sums selected raw records, without using the cleaning functions. Raw archives are downloaded unchanged; SHA-256 snapshots detect upstream revisions.

In [1]:
from pathlib import Path
import os, sys, subprocess
ROOT = Path.cwd()
if not (ROOT / "scripts/build_data.py").exists():
    if (ROOT.parent / "scripts/build_data.py").exists():
        ROOT = ROOT.parent
    else:
        ROOT = Path('/content/STEM-Pathways-Dashboard')
        if not ROOT.exists():
            subprocess.run(['git','clone','--branch','feature/stem-dashboard','https://github.com/APS-Data-Technologies/STEM-Pathways-Dashboard.git',str(ROOT)], check=True)
os.chdir(ROOT)
if str(ROOT) not in sys.path: sys.path.insert(0, str(ROOT))
try:
    import pandas as pd
    assert pd.__version__ == '2.2.3'
except (ImportError, AssertionError):
    subprocess.run([sys.executable,'-m','pip','install','pandas==2.2.3'],check=True)
print('Repository:', ROOT.name)


Repository: stem-pathways


## 1. Source snapshot and raw row counts

Download CYYYY_A.zip and HDYYYY.zip for each year. Prefer revised `_rv.csv` Completions members; fail if absent. UTF-8 with a Windows-1252 fallback handles older source files. The manifest contains original ZIP hashes, archive member, byte count and URL. No manual spreadsheet editing.

Download date is the fixed snapshot date, not a changing runtime value. Keep the branch available for Colab until the notebook's clone ref is changed to a reviewed release.

In [2]:
import hashlib, json
from scripts.build_data import download, read_zip, YEARS, FIELDS, AWARDS, build
expected_sources = json.loads((ROOT/'data/source-manifest.json').read_text())
for source in expected_sources:
    path = download(source['filename'])
    assert hashlib.sha256(path.read_bytes()).hexdigest() == source['sha256'], 'Source changed; review snapshot before rebuilding: '+source['filename']
print('All 16 source hashes match the committed snapshot.')
print('STEM families:', FIELDS)
print('Award levels:', AWARDS)


Source ready: C2016_A.zip
Source ready: HD2016.zip
Source ready: C2017_A.zip
Source ready: HD2017.zip
Source ready: C2018_A.zip
Source ready: HD2018.zip
Source ready: C2019_A.zip
Source ready: HD2019.zip
Source ready: C2020_A.zip
Source ready: HD2020.zip
Source ready: C2021_A.zip
Source ready: HD2021.zip
Source ready: C2022_A.zip
Source ready: HD2022.zip
Source ready: C2023_A.zip
Source ready: HD2023.zip
All 16 source hashes match the committed snapshot.
STEM families: {'11': 'Computer & information sciences', '14': 'Engineering', '15': 'Engineering technologies', '26': 'Biological & biomedical sciences', '27': 'Mathematics & statistics', '40': 'Physical sciences'}
Award levels: {3: 'Associate', 5: "Bachelor's", 7: "Master's"}


## 2. Cleaning steps and rationale

1. Read annual directory, enforce unique UNITID, keep Illinois. Join to same-year Completions (many-to-one) so historical/closed institutions are retained.
2. Keep `MAJORNUM=1`; discard second majors to avoid counting the same award twice.
3. Keep only six-digit `xx.xxxx` detail CIP codes. Do not sum detail with `99` or family subtotal rows.
4. Select six proposed STEM families and attach readable names.
5. Normalize padded `AWLEVEL` strings to numeric codes, then keep 3, 5, 7.
6. Use `CTOTALT`, require nonnegative integer counts and unique detail keys. Preserve reported zeros, do not impute absent records.
7. Group by year, UNITID, reported family and award level; verify summed counts unchanged by aggregation.
8. Build latest-known institution labels and year-presence history from all annual directories; sort exports deterministically. Each JSON and CSV must be under 2 MB.

Each stage's before/after counts are printed next and saved in `docs/qa/pipeline-audit.json`. No regression, forecasting, enrollment or salary data is added.

In [3]:
export_dir = ROOT/'public/data'
before = {p.name:hashlib.sha256(p.read_bytes()).hexdigest() for p in export_dir.iterdir()}
records, metadata, audit = build(download_sources=False)
display(pd.DataFrame(audit))
display(records.head(10))


2016: 302,043 raw rows -> 484 export rows; 22,744 awards
2017: 308,882 raw rows -> 486 export rows; 23,942 awards
2018: 290,149 raw rows -> 477 export rows; 23,969 awards
2019: 290,972 raw rows -> 464 export rows; 23,323 awards
2020: 289,833 raw rows -> 469 export rows; 22,270 awards
2021: 296,365 raw rows -> 473 export rows; 23,228 awards
2022: 301,055 raw rows -> 476 export rows; 22,667 awards
2023: 303,460 raw rows -> 479 export rows; 25,243 awards
{"rows": 3808, "institutions": 328, "total": 187386}


,year,raw_completions_rows,directory_rows,illinois_institutions,illinois_rows,first_major_rows,six_digit_rows,stem_rows,selected_award_rows,export_rows,total_completions,cip_version
0,2016,302043,7521,310,13702,10967,10226,1779,1121,484,22744,CIP 2010
1,2017,308882,7153,279,13751,11000,10278,1812,1149,486,23942,CIP 2010
2,2018,290149,6857,271,11974,11055,10351,1826,1161,477,23969,CIP 2010
3,2019,290972,6559,257,12023,11107,10428,1842,1164,464,23323,CIP 2010
4,2020,289833,6440,256,11930,11019,10296,1813,1198,469,22270,CIP 2020
5,2021,296365,6289,247,12147,11294,10569,1899,1227,473,23228,CIP 2020
6,2022,301055,6256,245,12439,11559,10837,1969,1259,476,22667,CIP 2020
7,2023,303460,6163,242,12651,11701,10968,2011,1279,479,25243,CIP 2020


,year,unitid,institution,field_family,field_name,award_level,completions
0,2016,142887,American Academy of Art,11,Computer & information sciences,Bachelor's,1
1,2016,143084,Augustana College,11,Computer & information sciences,Bachelor's,10
2,2016,143084,Augustana College,14,Engineering,Bachelor's,8
3,2016,143084,Augustana College,26,Biological & biomedical sciences,Bachelor's,119
4,2016,143084,Augustana College,27,Mathematics & statistics,Bachelor's,9
5,2016,143084,Augustana College,40,Physical sciences,Bachelor's,12
6,2016,143118,Aurora University,11,Computer & information sciences,Bachelor's,8
7,2016,143118,Aurora University,26,Biological & biomedical sciences,Bachelor's,24
8,2016,143118,Aurora University,27,Mathematics & statistics,Bachelor's,10
9,2016,143118,Aurora University,27,Mathematics & statistics,Master's,11


## Recorded row counts for this committed snapshot

| Award year | Raw completions rows | Illinois directory institutions | Export rows | Selected awards |
|---|---:|---:|---:|---:|
| 2016 | 302043 | 310 | 484 | 22744 |
| 2017 | 308882 | 279 | 486 | 23942 |
| 2018 | 290149 | 271 | 477 | 23969 |
| 2019 | 290972 | 257 | 464 | 23323 |
| 2020 | 289833 | 256 | 469 | 22270 |
| 2021 | 296365 | 247 | 473 | 23228 |
| 2022 | 301055 | 245 | 476 | 22667 |
| 2023 | 303460 | 242 | 479 | 25243 |

## 3. CIP transition — deliberately explicit limitation

CIP 2010 applies before 2020 and CIP 2020 thereafter. These are **as-reported family totals**, not a validated constant-classification series. The inventory below lists observed changes; appearance/disappearance is not itself proof of a crosswalk relationship. No unsupported one-to-one remapping is applied. The UI warns for ranges spanning 2020.

**B-18 remains partial:** verify the official NCES crosswalk and determine which changes cross family boundaries; obtain sponsor acceptance of as-reported series or implement/test the agreed harmonization. This notebook does not claim that unresolved methodological approval is complete.

Official reference: https://nces.ed.gov/ipeds/cipcode/crosswalk.aspx?y=56

In [4]:
transition = json.loads((ROOT/'docs/qa/cip-transition.json').read_text())
print(json.dumps(transition, indent=2))


{
  "note": "Observed additions/removals are not asserted to be one-to-one mappings. A sponsor-approved harmonized crosswalk is a remaining methodology review.",
  "added_2020": [
    "11.0204",
    "11.1005",
    "14.0102",
    "15.1301",
    "15.1701",
    "26.0903",
    "26.0908",
    "26.1199",
    "26.1307",
    "26.1309",
    "27.0502",
    "27.0601"
  ],
  "absent_2020": [
    "15.0503",
    "15.0702"
  ]
}


## 4. Independent number checks — 3 institutions × 2 years

The verifier uses Python CSV/ZIP directly, not pandas or the build functions. It sums six-digit first-major CTOTALT detail for the selected awards/families, then compares to exported totals. Judson, Northwestern and University of Chicago; 2016 and 2023. This is independent implementation-level verification; another teammate must still review the assumptions and repeat the checks.

In [5]:
from scripts.verify_data import verify
checks = verify()
display(pd.DataFrame(checks)[['institution','year','expected_source_total','export_total','status']])


[
  {
    "institution": "Judson University",
    "unitid": 146339,
    "year": 2016,
    "field": "All six STEM families",
    "award_level": "Associate + bachelor + master",
    "expected_source_total": 6,
    "export_total": 6,
    "status": "PASS",
    "source": "https://nces.ed.gov/ipeds/datacenter/data/C2016_A.zip",
    "member": "c2016_a_rv.csv"
  },
  {
    "institution": "Northwestern University",
    "unitid": 147767,
    "year": 2016,
    "field": "All six STEM families",
    "award_level": "Associate + bachelor + master",
    "expected_source_total": 1276,
    "export_total": 1276,
    "status": "PASS",
    "source": "https://nces.ed.gov/ipeds/datacenter/data/C2016_A.zip",
    "member": "c2016_a_rv.csv"
  },
  {
    "institution": "University of Chicago",
    "unitid": 144050,
    "year": 2016,
    "field": "All six STEM families",
    "award_level": "Associate + bachelor + master",
    "expected_source_total": 853,
    "export_total": 853,
    "status": "PASS",
    "source

,institution,year,expected_source_total,export_total,status
0,Judson University,2016,6,6,PASS
1,Northwestern University,2016,1276,1276,PASS
2,University of Chicago,2016,853,853,PASS
3,Judson University,2023,6,6,PASS
4,Northwestern University,2023,1846,1846,PASS
5,University of Chicago,2023,1480,1480,PASS


## 5. Exact-output reproducibility and size budget

Compare the fresh rebuild against the committed six files. Failure indicates source drift, dependency changes, contract changes or nondeterminism and must be investigated before committing. If intentionally refreshing the snapshot, document that change and update expected hashes through review.

In [6]:
after = {p.name:hashlib.sha256(p.read_bytes()).hexdigest() for p in export_dir.iterdir()}
assert before == after, 'Rebuild differs from committed export bytes'
assert len(after)==6
report = {'status':'PASS','files':after,'rows':len(records),'total_awards':int(records.completions.sum()),'source_hashes_checked':16}
(ROOT/'docs/qa/reproducibility.json').write_text(json.dumps(report, indent=2)+'\n')
print('PASS: all six exported files reproduce byte-for-byte.')
display(pd.DataFrame([{'file':p.name,'bytes':p.stat().st_size} for p in sorted(export_dir.iterdir())]))


PASS: all six exported files reproduce byte-for-byte.


,file,bytes
0,completions.csv,303806
1,completions.json,669304
2,institutions.csv,37703
3,institutions.json,63155
4,metadata.csv,5236
5,metadata.json,4737


## 6. Handover

Save the executed notebook, scripts, compact exports, source manifest and QA results through a feature-branch PR. Do not commit raw ZIPs, installed dependencies or credentials. Another teammate approves the PR. Read `docs/github-team-guide.md`. Frontend and Visualization must sign off the data contract; QA and Lead review About text; APS confirms open scope/CIP decisions.